In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kanchandalal123/mcq-ranking-train-dataset-2/ranking_train (1).csv
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver - DeBERTa Baseline

Goal:

- Fine-tune DeBERTa-v3-base
- Train on ranking dataset
- Evaluate using GroupKFold
- Generate first submission

In [2]:
!pip install -q transformers datasets accelerate

In [3]:
import gc
import os
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from sklearn.model_selection import GroupKFold

import wandb

In [4]:
df = pd.read_csv(
    "/kaggle/input/datasets/kanchandalal123/mcq-ranking-train-dataset-2/ranking_train (1).csv"
)

df.head()

,id,question,option_label,option_text,label,text,text_length,fold
0,1,Pick the best possible answer: What is Martin ...,A,Martin Heidegger believes that humans exist wi...,0,Question: Pick the best possible answer: What ...,474,3
1,1,Pick the best possible answer: What is Martin ...,B,Martin Heidegger believes that humans do not e...,1,Question: Pick the best possible answer: What ...,425,3
2,1,Pick the best possible answer: What is Martin ...,C,Martin Heidegger does not believe in the exist...,0,Question: Pick the best possible answer: What ...,389,3
3,1,Pick the best possible answer: What is Martin ...,D,Martin Heidegger believes that the relationshi...,0,Question: Pick the best possible answer: What ...,369,3
4,1,Pick the best possible answer: What is Martin ...,E,Martin Heidegger believes that time is an illu...,0,Question: Pick the best possible answer: What ...,358,3


In [5]:
wandb.init(
    project="24f1002360-t22026",
    name="deberta_baseline_v2_fp32",
    config={
        "model":"deberta-v3-base",
        "dtype":"float32",
        "epochs":2,
        "lr":1e-5,
        "batch_size":8
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: 24f1002360 (24f1002360-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
MODEL_NAME = "microsoft/deberta-v3-base"

MAX_LENGTH = 256

FOLD = 0

In [7]:
train_df = df[
    df["fold"] != FOLD
].reset_index(drop=True)

valid_df = df[
    df["fold"] == FOLD
].reset_index(drop=True)

valid_metadata = valid_df[
    ["id","option_label","label"]
].copy()

valid_metadata = valid_metadata.reset_index(
    drop=True
)

print(train_df.shape)
print(valid_df.shape)

(8000, 8)
(2000, 8)


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [9]:
def tokenize(example):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        max_length=256
    )

    encoded.pop("token_type_ids", None)

    return encoded

In [10]:
train_ds = Dataset.from_pandas(
    train_df[["text","label"]]
)

valid_ds = Dataset.from_pandas(
    valid_df[["text","label"]]
)

In [11]:
train_ds = train_ds.map(
    tokenize,
    batched=True
)

valid_ds = valid_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [12]:
train_ds = train_ds.remove_columns(
    ["text"]
)

valid_ds = valid_ds.remove_columns(
    ["text"]
)

train_ds.set_format("torch")
valid_ds.set_format("torch")

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias          

In [15]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [16]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

import torch
import torch.nn.functional as F

def apk(actual, predicted, k=3):

    if actual in predicted[:k]:
        return 1.0 / (
            predicted[:k].index(actual) + 1
        )

    return 0.0


def mapk(actuals, predictions, k=3):

    return np.mean([
        apk(a,p,k)
        for a,p in zip(
            actuals,
            predictions
        )
    ])


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=1
    )

    acc = accuracy_score(
        labels,
        preds
    )

    f1 = f1_score(
        labels,
        preds
    )

    scores = F.softmax(
        torch.tensor(logits),
        dim=1
    )[:,1].numpy()

    temp_df = valid_metadata.copy()

    temp_df["score"] = scores

    actuals = []

    predictions = []

    for qid, grp in temp_df.groupby("id"):

        grp = grp.sort_values(
            "score",
            ascending=False
        )

        top3 = grp[
            "option_label"
        ].head(3).tolist()

        actual = grp[
            grp["label"] == 1
        ]["option_label"].iloc[0]

        actuals.append(actual)

        predictions.append(top3)

    map3 = mapk(
        actuals,
        predictions,
        k=3
    )

    return {
        "accuracy": acc,
        "f1": f1,
        "map3": map3
    }

In [17]:
training_args = TrainingArguments(
    output_dir="deberta_baseline",

    learning_rate=1e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    metric_for_best_model="map3",
    greater_is_better=True,

    load_best_model_at_end=True,

    logging_steps=25,

    report_to="wandb",

    fp16=False,
    bf16=False
)

In [18]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=valid_ds,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [20]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Map3
1,1.022035,0.967650,0.800000,0.000000,0.737917
2,0.745724,0.717606,0.851000,0.520900,0.889583


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1000, training_loss=0.9486916656494141, metrics={'train_runtime': 435.436, 'train_samples_per_second': 36.745, 'train_steps_per_second': 2.297, 'total_flos': 877902224526720.0, 'train_loss': 0.9486916656494141, 'epoch': 2.0})

In [21]:
results = trainer.evaluate()

results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.7176055908203125,
 'eval_accuracy': 0.851,
 'eval_f1': 0.5209003215434084,
 'eval_map3': 0.8895833333333334,
 'eval_runtime': 17.0458,
 'eval_samples_per_second': 117.331,
 'eval_steps_per_second': 7.333,
 'epoch': 2.0}

In [30]:
trainer.save_model(
    "/kaggle/working/deberta_baseline_model"
)

tokenizer.save_pretrained(
    "/kaggle/working/deberta_baseline_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/deberta_baseline_model/tokenizer_config.json',
 '/kaggle/working/deberta_baseline_model/tokenizer.json')

In [31]:
metrics_df = pd.DataFrame(
    [results]
)

metrics_df.to_csv(
    "/kaggle/working/deberta_baseline_metrics.csv",
    index=False
)

metrics_df

,eval_loss,eval_accuracy,eval_f1,eval_map3,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.717606,0.851,0.5209,0.889583,17.0458,117.331,7.333,2.0


In [32]:
wandb.finish()

eval/accuracy,▁▇██
eval/f1,▅▁██
eval/loss,█▃▁▁
eval/map3,▁▆██
eval/runtime,█▁▁▂
eval/samples_per_second,▁██▇
eval/steps_per_second,▁██▇
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
+5,...


In [44]:
import shutil

shutil.make_archive(
    "/kaggle/working/deberta_baseline_model",
    "zip",
    "/kaggle/working/deberta_baseline_model"
)

print("ZIP Created")

ZIP Created
